# 📥 RevertIQ — Data Pipeline

This notebook demonstrates how to:
1. Download OHLCV data for NIFTY 50 stocks
2. Download index data (NIFTY 50, India VIX)
3. Clean and validate the data
4. Store and reload processed data

In [ ]:
# ── Setup (Colab) ──
# !git clone https://github.com/username/RevertIQ.git /content/RevertIQ
# %cd /content/RevertIQ
# !pip install -e . -q

import warnings
warnings.filterwarnings('ignore')

from revertiq.config.settings import get_default_config
from revertiq.data.universe import get_nifty50_tickers, get_index_tickers, validate_universe
from revertiq.data.downloader import DataDownloader
from revertiq.data.cleaner import DataCleaner

import pandas as pd
print("Modules loaded ✓")

## 1. Explore the Universe

In [ ]:
# NIFTY 50 constituent tickers
tickers = get_nifty50_tickers()
print(f"Total tickers: {len(tickers)}")
print(f"\nFull list:")
for i, t in enumerate(tickers, 1):
    print(f"  {i:2d}. {t}")

In [ ]:
# Index tickers
indices = get_index_tickers()
print("Index tickers:")
for name, ticker in indices.items():
    print(f"  {name:12s} → {ticker}")

In [ ]:
# Optional: validate against Wikipedia
# diff = validate_universe()
# print(f"Added (new on Wikipedia): {diff['added']}")
# print(f"Removed (not on Wikipedia): {diff['removed']}")

## 2. Download Data

In [ ]:
config = get_default_config()
print(f"Date range: {config.data.start_date} → {config.data.end_date}")
print(f"Chunk size: {config.data.chunk_size} tickers/batch")
print(f"Delay: {config.data.delay_between_chunks}s between batches")

downloader = DataDownloader(config.data)

In [ ]:
# Download all stocks (2-5 minutes)
raw_data = downloader.download_all()
print(f"\nSuccessfully downloaded: {len(raw_data)} tickers")

In [ ]:
# Download index data
index_data = downloader.download_index_data()
for name, df in index_data.items():
    if not df.empty:
        print(f"{name}: {len(df)} rows ({df.index[0].date()} → {df.index[-1].date()})")
    else:
        print(f"{name}: No data (may be unavailable)")

In [ ]:
# Inspect a sample ticker
sample = list(raw_data.values())[0]
print(f"\nSample data shape: {sample.shape}")
print(f"Columns: {list(sample.columns)}")
sample.tail()

## 3. Clean Data

In [ ]:
cleaner = DataCleaner(config.data)
clean_data = cleaner.clean_all()

report = cleaner.get_cleaning_report()
print(f"\nCleaning Report:")
for k, v in report.items():
    print(f"  {k}: {v}")

In [ ]:
# Combined DataFrame for all stocks
combined = cleaner.get_combined_df()
print(f"\nCombined shape: {combined.shape}")
print(f"Unique tickers: {combined['ticker'].nunique()}")
print(f"Date range: {combined['date'].min().date()} → {combined['date'].max().date()}")
combined.head(10)

## 4. Data Quality Checks

In [ ]:
# Check for any remaining NaNs
nan_pct = combined.isnull().mean() * 100
print("NaN % by column:")
print(nan_pct[nan_pct > 0].to_string())
if nan_pct.max() == 0:
    print("  No NaN values ✓")

In [ ]:
# Data coverage per ticker
coverage = combined.groupby('ticker').size().describe()
print("\nData points per ticker:")
print(coverage)

In [ ]:
# Quick price chart
import plotly.express as px

sample_tickers = combined['ticker'].unique()[:5]
sample_data = combined[combined['ticker'].isin(sample_tickers)]
fig = px.line(sample_data, x='date', y='close', color='ticker',
              title='Sample Stock Prices')
fig.update_layout(template='plotly_dark',
                  paper_bgcolor='#161b22', plot_bgcolor='#0d1117')
fig.show()

## 5. Reload Saved Data

On subsequent runs you can skip the download step:

In [ ]:
# Load from disk (skip download)
# cleaner = DataCleaner(config.data)
# clean_data = cleaner.load_all_processed()
# combined = cleaner.get_combined_df()
# print(f"Loaded {len(clean_data)} tickers from disk")